[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/32.%20The%20Inter-Terminal%20Transfer%20Optimization%20Problem/P32-Tier-5_executed.ipynb)

---


# 32. The Inter-Terminal Transfer Optimization Problem

## Tier 5: Integrated Digital Twin Simulation

### Goal

This notebook builds a small, simulated digital-twin architecture for inter-terminal container transfers. It combines simulated sensor streams, a changing operational state, simple demand and congestion forecasts, bottleneck detection, alerts, and KPI visualizations. The aim is to show how these components can be connected in a system-level simulation.

The code is an educational simulation, not a live connection to a port and not a validated operational digital twin. It does not solve an optimization model or establish that a particular operating policy is optimal. Its outputs describe the behavior of the assumptions and random processes implemented here; they are not measured performance claims.

### Inputs and outputs

- **Inputs:** Five terminal identifiers, six vehicle capacity and cost specifications, a terminal-to-terminal distance matrix, and the model assumptions defined in the code.
- **Outputs:** Simulated sensor records, container and vehicle states, demand and congestion forecasts, alerts, bottleneck indicators, KPIs, and plots.
- **Time scale:** The baseline example advances the simulation in one-minute steps over a simulated 24-hour period.

### Modeling assumptions and scope

- Sensor readings, vehicle state changes, demand, and congestion are generated by simplified stochastic rules. They are not calibrated against operational data.
- A container is considered assigned when a suitable active vehicle is selected. An assignment remains in the simulation for up to one hour; this bookkeeping rule is not a physical travel or delivery model.
- The reported on-time measure is the fraction of assigned containers assigned within 30 minutes of becoming ready. It is not a delivery-time measure.
- The cost calculation is a simplified model cost, not a complete financial accounting model. Its units depend on the illustrative vehicle cost inputs.
- The code does not model detailed vehicle motion, battery use, container handling physics, live data synchronization, or a validated predictive model. Forecasts are illustrative and are not evaluated against observed outcomes.

### Integrated simulation workflow

The implementation follows a simulation-and-monitoring sequence: initialize the state and sensors; collect sensor records when their update intervals have elapsed; advance vehicle, terminal, and demand states; assign containers to vehicles subject to activity and capacity; calculate KPIs; and check for alerts and bottlenecks. This sequence connects the subsystems without introducing a separate optimization solver.

### Interpreting results

Read the printed KPIs and plots as descriptions of this simulated run. Because the simulation uses random draws and illustrative assumptions, one run is not evidence of predictive accuracy, real-world improvement, return on investment, or system resilience. Such claims would require suitable operational data, validation, and an explicitly defined comparison study.

### When this modeling tier is useful

A system-level simulation can help organize questions about data integration, monitoring, and interactions among terminals, vehicles, and demand. In a real deployment, its value would depend on the quality of the connected data, the fidelity and validation of the models, and how decision-makers use the results. More terminals, sensors, or model components do not by themselves guarantee better operational decisions.

In [ ]:
# Numerical, plotting, timing, and state-history tools used by the simulation.
import random
import time
from collections import defaultdict, deque
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class IoTDataSensor:
    """Generate illustrative sensor records from the current simulated state."""

    def __init__(self, sensor_id, sensor_type, location, update_interval=30):
        self.sensor_id = sensor_id
        self.sensor_type = sensor_type
        self.location = location
        self.update_interval = update_interval
        self.last_update = time.time()
        self.data_history = deque(maxlen=100)
        self.noise_level = 0.05
        self.failure_rate = 0.01
        self.is_active = True

    def generate_data(self, current_time, system_state):
        """Return one reading, or None when the sensor is inactive or fails."""
        if not self.is_active:
            return None

        # A failed sensor stops reporting, making data availability part of the simulation.
        if random.random() < self.failure_rate:
            self.is_active = False
            return None

        data = {
            "sensor_id": self.sensor_id,
            "sensor_type": self.sensor_type,
            "location": self.location,
            "timestamp": current_time,
            "data": None,
        }

        if self.sensor_type == "container_position":
            containers = system_state.get("containers", [])
            if containers:
                container = random.choice(containers)
                base_position = system_state["terminal_positions"].get(
                    container["destination"], (0, 0)
                )
                noise = np.random.normal(0, self.noise_level, 2)
                data["data"] = {
                    "container_id": container["id"],
                    "position": (
                        base_position[0] + noise[0],
                        base_position[1] + noise[1],
                    ),
                    "status": "assigned" if container["assigned"] else "awaiting_assignment",
                }

        elif self.sensor_type == "vehicle_gps":
            vehicles = system_state.get("vehicles", [])
            vehicle_states = system_state.get("vehicle_states", {})
            if vehicles:
                vehicle = random.choice(vehicles)
                state = vehicle_states[vehicle.id]
                base_position = system_state["terminal_positions"].get(
                    state["position"], (0, 0)
                )
                movement = np.random.normal(0, 0.1, 2)
                data["data"] = {
                    "vehicle_id": vehicle.id,
                    "position": (
                        base_position[0] + movement[0],
                        base_position[1] + movement[1],
                    ),
                    "speed": random.uniform(5, 25),
                    "heading": random.uniform(0, 360),
                    "fuel_level": state["fuel_level"],
                }

        elif self.sensor_type == "terminal_congestion":
            terminal_id = self.location
            congestion = random.uniform(0.2, 0.9)
            data["data"] = {
                "terminal_id": terminal_id,
                "congestion_level": congestion,
                "queue_length": int(congestion * 50),
                "processing_rate": random.uniform(8, 15),
                "estimated_wait_time": congestion * 30,
            }

        elif self.sensor_type == "weather":
            data["data"] = {
                "temperature": random.uniform(10, 35),
                "humidity": random.uniform(30, 90),
                "wind_speed": random.uniform(0, 25),
                "visibility": random.uniform(1, 10),
                "precipitation": random.choice(
                    ["none", "light", "moderate", "heavy"]
                ),
                "impact_factor": random.uniform(0.8, 1.2),
            }

        if data["data"] is not None:
            self.data_history.append(data)
            self.last_update = current_time
            return data

        return None


class PredictiveAnalytics:
    """Provide simple illustrative demand and congestion forecasts."""

    def __init__(self, history_window=24):
        self.history_window = history_window
        self.demand_history = deque(maxlen=history_window * 60)
        self.congestion_history = deque(maxlen=history_window * 60)
        self.weather_history = deque(maxlen=history_window * 60)

    def add_demand_data(self, timestamp, demand_level):
        self.demand_history.append(
            {"timestamp": timestamp, "demand": demand_level}
        )

    def add_congestion_data(self, timestamp, terminal_id, congestion_level):
        self.congestion_history.append(
            {
                "timestamp": timestamp,
                "terminal_id": terminal_id,
                "congestion": congestion_level,
            }
        )

    def add_weather_data(self, timestamp, weather_data):
        self.weather_history.append(
            {"timestamp": timestamp, "weather": weather_data}
        )

    def forecast_demand(self, forecast_hours=4):
        """Combine recent demand, a recent trend, and an hourly seasonal profile."""
        if len(self.demand_history) < 60:
            return [50.0] * forecast_hours

        recent_demands = [
            record["demand"] for record in list(self.demand_history)[-60:]
        ]
        recent_avg = np.mean(recent_demands[-30:])
        older_avg = np.mean(recent_demands[:30])
        trend = (recent_avg - older_avg) / older_avg if older_avg > 0 else 0

        latest_timestamp = self.demand_history[-1]["timestamp"]
        current_hour = datetime.fromtimestamp(latest_timestamp).hour
        seasonal_factors = {
            0: 0.3, 1: 0.2, 2: 0.1, 3: 0.1, 4: 0.2, 5: 0.4,
            6: 0.7, 7: 0.9, 8: 1.0, 9: 1.0, 10: 0.8, 11: 0.6,
            12: 0.7, 13: 0.8, 14: 0.9, 15: 1.0, 16: 1.1, 17: 1.2,
            18: 1.1, 19: 1.0, 20: 0.8, 21: 0.6, 22: 0.5, 23: 0.4,
        }

        base_demand = np.mean(recent_demands)
        forecast = []

        for hour in range(forecast_hours):
            forecast_hour = (current_hour + hour) % 24
            seasonal_factor = seasonal_factors.get(forecast_hour, 1.0)
            noise = np.random.normal(0, 0.1)
            trend_factor = 1 + trend * hour / max(1, forecast_hours)
            predicted_demand = (
                base_demand * seasonal_factor * trend_factor * (1 + noise)
            )
            forecast.append(max(10.0, predicted_demand))

        return forecast

    def predict_congestion(self, terminal_id, forecast_hours=4):
        """Project congestion using its recent level and the demand forecast."""
        terminal_congestion = [
            record["congestion"]
            for record in self.congestion_history
            if record["terminal_id"] == terminal_id
        ]

        if len(terminal_congestion) < 30:
            return [0.5] * forecast_hours

        recent_congestion = np.mean(terminal_congestion[-30:])
        demand_forecast = self.forecast_demand(forecast_hours)
        demand_factor = np.mean(demand_forecast) / 50.0
        forecast = []

        for _ in range(forecast_hours):
            predicted_congestion = (
                recent_congestion * 0.7 + demand_factor * 0.3
            )
            predicted_congestion += np.random.normal(0, 0.05)
            predicted_congestion = max(0.1, min(0.95, predicted_congestion))
            forecast.append(predicted_congestion)
            recent_congestion = predicted_congestion

        return forecast

    def identify_bottlenecks(self):
        """Flag recent high congestion and unusually high demand observations."""
        bottlenecks = []

        if self.congestion_history:
            recent_congestion = defaultdict(list)
            latest_timestamp = self.congestion_history[-1]["timestamp"]
            cutoff_time = latest_timestamp - 300

            for record in self.congestion_history:
                if record["timestamp"] > cutoff_time:
                    recent_congestion[record["terminal_id"]].append(
                        record["congestion"]
                    )

            for terminal_id, levels in recent_congestion.items():
                avg_congestion = np.mean(levels)
                if avg_congestion > 0.7:
                    bottlenecks.append(
                        {
                            "type": "congestion",
                            "location": terminal_id,
                            "severity": avg_congestion,
                            "predicted_duration": 1,
                        }
                    )

        if len(self.demand_history) > 60:
            recent_demand = [
                record["demand"] for record in list(self.demand_history)[-60:]
            ]
            threshold = np.mean(recent_demand) + 2 * np.std(recent_demand)

            for record in list(self.demand_history)[-10:]:
                if record["demand"] > threshold:
                    bottlenecks.append(
                        {
                            "type": "demand_spike",
                            "severity": record["demand"] / threshold,
                            "timestamp": record["timestamp"],
                        }
                    )

        return bottlenecks

In [ ]:
class InterTerminalDigitalTwin:
    """Coordinate the simulated state, sensors, analytics, and KPI reporting."""

    def __init__(self, terminals, vehicles, distances):
        self.terminals = terminals
        self.vehicles = vehicles
        self.distances = np.asarray(distances, dtype=float)

        if self.distances.shape != (len(terminals), len(terminals)):
            raise ValueError("The distance matrix must have one row and column per terminal.")
        if np.any(self.distances < 0):
            raise ValueError("Distances must be non-negative.")

        self.terminal_positions = {
            "A": (0, 0),
            "B": (3, 0),
            "C": (1.5, 2.5),
            "D": (6, 0),
            "E": (4.5, 2.5),
        }

        self.sensors = []
        self._initialize_sensors()

        self.analytics = PredictiveAnalytics()
        self.current_time = time.time()
        self.containers = []
        self.vehicle_states = {}
        self.terminal_states = {}
        self.kpi_history = deque(maxlen=1000)
        self.alert_history = deque(maxlen=100)
        self.simulation_speed = 1.0
        self.data_quality = 0.95

        self._initialize_system_state()

    def _initialize_sensors(self):
        sensor_id = 0

        for terminal in self.terminals:
            for _ in range(3):
                self.sensors.append(
                    IoTDataSensor(
                        sensor_id=sensor_id,
                        sensor_type="container_position",
                        location=terminal,
                        update_interval=30,
                    )
                )
                sensor_id += 1

        for vehicle in self.vehicles:
            self.sensors.append(
                IoTDataSensor(
                    sensor_id=sensor_id,
                    sensor_type="vehicle_gps",
                    location=vehicle.id,
                    update_interval=10,
                )
            )
            sensor_id += 1

        for terminal in self.terminals:
            self.sensors.append(
                IoTDataSensor(
                    sensor_id=sensor_id,
                    sensor_type="terminal_congestion",
                    location=terminal,
                    update_interval=60,
                )
            )
            sensor_id += 1

        for _ in range(2):
            self.sensors.append(
                IoTDataSensor(
                    sensor_id=sensor_id,
                    sensor_type="weather",
                    location="port_area",
                    update_interval=300,
                )
            )
            sensor_id += 1

    def _initialize_system_state(self):
        for vehicle in self.vehicles:
            self.vehicle_states[vehicle.id] = {
                "position": random.choice(self.terminals),
                "load": random.uniform(0, vehicle.capacity * 0.7),
                "fuel_level": random.uniform(40, 90),
                "status": "active",
                "last_maintenance": time.time() - random.uniform(0, 86400 * 30),
                "total_distance": random.uniform(100, 1000),
                "assignments_completed": random.randint(5, 50),
            }

        for terminal in self.terminals:
            self.terminal_states[terminal] = {
                "queue_length": random.randint(0, 30),
                "processing_rate": random.uniform(8, 15),
                "utilization": random.uniform(0.3, 0.8),
                "equipment_status": "operational",
                "staff_available": random.randint(5, 20),
            }

    def collect_sensor_data(self):
        sensor_data = []
        system_state = self._get_system_state()

        for sensor in self.sensors:
            elapsed = self.current_time - sensor.last_update
            if elapsed >= sensor.update_interval:
                data = sensor.generate_data(self.current_time, system_state)
                if data is None:
                    continue

                sensor_data.append(data)
                if sensor.sensor_type == "terminal_congestion":
                    self.analytics.add_congestion_data(
                        data["timestamp"],
                        data["data"]["terminal_id"],
                        data["data"]["congestion_level"],
                    )
                elif sensor.sensor_type == "weather":
                    self.analytics.add_weather_data(
                        data["timestamp"], data["data"]
                    )

        return sensor_data

    def _get_system_state(self):
        return {
            "terminals": self.terminals,
            "vehicles": self.vehicles,
            "vehicle_states": self.vehicle_states,
            "terminal_positions": self.terminal_positions,
            "containers": self.containers,
        }

    def update_simulation(self, time_step=60):
        if time_step <= 0:
            raise ValueError("The simulation time step must be positive.")

        self.current_time += time_step
        sensor_data = self.collect_sensor_data()
        self._update_vehicle_states()
        self._update_terminal_states()
        self._generate_container_demand(time_step)
        self._process_transfers()

        kpis = self._calculate_kpis()
        self.kpi_history.append(
            {"timestamp": self.current_time, "kpis": kpis}
        )

        alerts = self._check_alerts(kpis)
        self.alert_history.extend(alerts)

        return {
            "timestamp": self.current_time,
            "sensor_data": sensor_data,
            "kpis": kpis,
            "alerts": alerts,
            "bottlenecks": self.analytics.identify_bottlenecks(),
        }

    def _update_vehicle_states(self):
        terminal_indices = {
            terminal: index for index, terminal in enumerate(self.terminals)
        }

        for vehicle in self.vehicles:
            state = self.vehicle_states[vehicle.id]

            if state["status"] == "active":
                if random.random() < 0.1:
                    old_position = state["position"]
                    new_position = random.choice(self.terminals)
                    distance = self.distances[
                        terminal_indices[old_position],
                        terminal_indices[new_position],
                    ]
                    state["position"] = new_position
                    state["total_distance"] += distance

                state["fuel_level"] = max(
                    0, state["fuel_level"] - random.uniform(0.1, 0.5)
                )
                if state["fuel_level"] < 20:
                    state["status"] = "needs_fuel"

                if random.random() < 0.3:
                    state["load"] += random.uniform(-1, 2)
                    state["load"] = max(0, min(vehicle.capacity, state["load"]))
                    state["assignments_completed"] += 1

    def _update_terminal_states(self):
        for terminal in self.terminals:
            state = self.terminal_states[terminal]

            demand_factor = 1.0
            if self.analytics.demand_history:
                recent_demand = np.mean(
                    [record["demand"] for record in list(self.analytics.demand_history)[-10:]]
                )
                demand_factor = recent_demand / 50.0

            state["queue_length"] = int(
                state["queue_length"] * demand_factor + random.uniform(-5, 5)
            )
            state["queue_length"] = max(0, min(100, state["queue_length"]))
            state["utilization"] = min(
                0.95,
                max(0, state["queue_length"] / 50.0 + random.uniform(-0.1, 0.1)),
            )
            state["processing_rate"] = max(
                5, state["processing_rate"] + random.uniform(-1, 1)
            )

    def _generate_container_demand(self, time_step=60):
        base_demand = 50
        current_hour = datetime.fromtimestamp(self.current_time).hour
        hourly_multipliers = {
            0: 0.3, 1: 0.2, 2: 0.1, 3: 0.1, 4: 0.2, 5: 0.4,
            6: 0.7, 7: 0.9, 8: 1.0, 9: 1.0, 10: 0.8, 11: 0.6,
            12: 0.7, 13: 0.8, 14: 0.9, 15: 1.0, 16: 1.1, 17: 1.2,
            18: 1.1, 19: 1.0, 20: 0.8, 21: 0.6, 22: 0.5, 23: 0.4,
        }

        multiplier = hourly_multipliers.get(current_hour, 1.0)
        demand_rate = base_demand * multiplier / 3600
        num_containers = np.random.poisson(demand_rate * time_step)

        # One observation per time step keeps the demand history on the simulation's time scale.
        self.analytics.add_demand_data(self.current_time, num_containers)

        for _ in range(num_containers):
            origin = random.choice(self.terminals)
            destination = random.choice(
                [terminal for terminal in self.terminals if terminal != origin]
            )
            self.containers.append(
                {
                    "id": len(self.containers) + 1,
                    "origin": origin,
                    "destination": destination,
                    "weight": random.choice([1, 2, 3]),
                    "priority": random.choice(["low", "medium", "high"]),
                    "ready_time": self.current_time,
                    "assigned": False,
                    "vehicle_id": None,
                }
            )

    def _process_transfers(self):
        unassigned = [container for container in self.containers if not container["assigned"]]

        # The per-step limit bounds assignment work in this illustrative simulation.
        for container in unassigned[:10]:
            suitable_vehicles = []
            for vehicle in self.vehicles:
                state = self.vehicle_states[vehicle.id]
                if (
                    state["status"] == "active"
                    and state["load"] + container["weight"] <= vehicle.capacity
                ):
                    suitable_vehicles.append(vehicle)

            if suitable_vehicles:
                chosen_vehicle = random.choice(suitable_vehicles)
                container["assigned"] = True
                container["vehicle_id"] = chosen_vehicle.id
                container["assigned_time"] = self.current_time
                self.vehicle_states[chosen_vehicle.id]["load"] += container["weight"]
                self.vehicle_states[chosen_vehicle.id]["assignments_completed"] += 1

        # Assigned records are retained for a one-hour bookkeeping window, not as modeled deliveries.
        self.containers = [
            container
            for container in self.containers
            if not container["assigned"]
            or self.current_time - container["ready_time"] < 3600
        ]

    def _calculate_kpis(self):
        total_containers = len(self.containers)
        assigned_containers = sum(
            1 for container in self.containers if container["assigned"]
        )
        transfer_efficiency = assigned_containers / max(1, total_containers)

        total_capacity = sum(vehicle.capacity for vehicle in self.vehicles)
        total_load = sum(
            self.vehicle_states[vehicle.id]["load"] for vehicle in self.vehicles
        )
        vehicle_utilization = total_load / total_capacity if total_capacity else 0.0

        assigned_waits = [
            (container["assigned_time"] - container["ready_time"]) / 60
            for container in self.containers
            if container["assigned"]
        ]
        avg_transfer_time = float(np.mean(assigned_waits)) if assigned_waits else 0.0

        fuel_cost = sum(
            self.vehicle_states[vehicle.id]["total_distance"]
            * vehicle.cost_per_km
            for vehicle in self.vehicles
        )
        labor_cost = len(self.vehicles) * 50
        total_cost = fuel_cost + labor_cost

        on_time_assignments = sum(
            1
            for container in self.containers
            if container["assigned"]
            and container["assigned_time"] - container["ready_time"] < 1800
        )
        on_time_assignment_rate = on_time_assignments / max(1, assigned_containers)

        return {
            "transfer_efficiency": transfer_efficiency,
            "vehicle_utilization": vehicle_utilization,
            "avg_transfer_time": avg_transfer_time,
            "operational_cost": total_cost,
            "on_time_assignment_rate": on_time_assignment_rate,
            "total_containers": total_containers,
            "assigned_containers": assigned_containers,
        }

    def _check_alerts(self, kpis):
        alerts = []

        if kpis["transfer_efficiency"] < 0.7:
            alerts.append(
                {
                    "type": "warning",
                    "message": f"Low assignment efficiency: {kpis['transfer_efficiency']:.1%}",
                    "severity": "medium",
                    "timestamp": self.current_time,
                }
            )

        if kpis["vehicle_utilization"] > 0.9:
            alerts.append(
                {
                    "type": "warning",
                    "message": f"High vehicle utilization: {kpis['vehicle_utilization']:.1%}",
                    "severity": "high",
                    "timestamp": self.current_time,
                }
            )

        if kpis["on_time_assignment_rate"] < 0.85:
            alerts.append(
                {
                    "type": "critical",
                    "message": (
                        "Low on-time assignment rate: "
                        f"{kpis['on_time_assignment_rate']:.1%}"
                    ),
                    "severity": "high",
                    "timestamp": self.current_time,
                }
            )

        return alerts

In [ ]:
def create_digital_twin_simulation():
    """Create the illustrative terminal, fleet, and distance inputs."""
    terminals = ["A", "B", "C", "D", "E"]

    vehicles = [
        type("Vehicle", (), {"id": 1, "capacity": 3, "cost_per_km": 10}),
        type("Vehicle", (), {"id": 2, "capacity": 2, "cost_per_km": 12}),
        type("Vehicle", (), {"id": 3, "capacity": 4, "cost_per_km": 8}),
        type("Vehicle", (), {"id": 4, "capacity": 3, "cost_per_km": 11}),
        type("Vehicle", (), {"id": 5, "capacity": 2, "cost_per_km": 13}),
        type("Vehicle", (), {"id": 6, "capacity": 5, "cost_per_km": 9}),
    ]

    distance_matrix = np.array(
        [
            [0, 3, 5, 7, 6],
            [3, 0, 4, 6, 5],
            [5, 4, 0, 4, 3],
            [7, 6, 4, 0, 3],
            [6, 5, 3, 3, 0],
        ]
    )

    return InterTerminalDigitalTwin(terminals, vehicles, distance_matrix)


digital_twin = create_digital_twin_simulation()

print("=== DIGITAL TWIN SIMULATION INITIALIZATION ===")
print(f"Terminals: {digital_twin.terminals}")
print(f"Vehicles: {len(digital_twin.vehicles)}")
print(f"Sensors: {len(digital_twin.sensors)}")
print(
    "Sensor types: "
    f"{sorted({sensor.sensor_type for sensor in digital_twin.sensors})}"
)
print("Simulation components initialized.")
print(f"Analytics component: {type(digital_twin.analytics).__name__}")

In [ ]:
# Run the baseline scenario for 24 simulated hours using one-minute steps.
print("=== BASELINE SIMULATION (24-HOUR TRIAL) ===")
print("Advancing the illustrative system state in one-minute steps.\n")

baseline_kpis = []
time_steps = 24 * 60

for step in range(time_steps):
    result = digital_twin.update_simulation(60)
    baseline_kpis.append(result["kpis"])

    if step % 60 == 0:
        print(
            f"Hour {step // 60 + 1}/24: "
            f"Assignment efficiency: {result['kpis']['transfer_efficiency']:.1%}, "
            f"Vehicle utilization: {result['kpis']['vehicle_utilization']:.1%}, "
            f"On-time assignment: "
            f"{result['kpis']['on_time_assignment_rate']:.1%}"
        )

final_kpis = baseline_kpis[-1]
avg_kpis = {
    key: float(np.mean([kpi[key] for kpi in baseline_kpis]))
    for key in baseline_kpis[0]
}
baseline_avg = avg_kpis

print("\n=== BASELINE SUMMARY ===")
print(f"Mean assignment efficiency: {avg_kpis['transfer_efficiency']:.1%}")
print(f"Mean vehicle utilization: {avg_kpis['vehicle_utilization']:.1%}")
print(f"Mean on-time assignment rate: {avg_kpis['on_time_assignment_rate']:.1%}")
print(f"Mean assignment waiting time: {avg_kpis['avg_transfer_time']:.1f} minutes")
print(f"Mean modeled operational cost: {avg_kpis['operational_cost']:.1f} model cost units")
print(
    "Interpret these values as outputs of this stochastic simulation, "
    "not as measured port performance."
)

In [ ]:
def visualize_digital_twin_results(kpi_history, average_kpis):
    """Plot KPI trajectories and a summary of the simulated baseline run."""
    if not kpi_history:
        raise ValueError("At least one KPI record is required for visualization.")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Illustrative Digital Twin Simulation Results",
        fontsize=16,
        fontweight="bold",
    )

    time_hours = np.arange(len(kpi_history)) / 60

    ax1 = axes[0, 0]
    ax1.plot(
        time_hours,
        [kpi["transfer_efficiency"] * 100 for kpi in kpi_history],
        label="Assignment efficiency",
        linewidth=2,
    )
    ax1.plot(
        time_hours,
        [kpi["vehicle_utilization"] * 100 for kpi in kpi_history],
        label="Vehicle utilization",
        linewidth=2,
    )
    ax1.plot(
        time_hours,
        [kpi["on_time_assignment_rate"] * 100 for kpi in kpi_history],
        label="On-time assignment rate",
        linewidth=2,
    )
    ax1.set_xlabel("Simulated time (hours)")
    ax1.set_ylabel("Rate (%)")
    ax1.set_title("KPI Trajectories")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 100)

    ax2 = axes[0, 1]
    operational_costs = [kpi["operational_cost"] for kpi in kpi_history]
    ax2.plot(time_hours, operational_costs, color="darkorange", linewidth=2)
    ax2.set_xlabel("Simulated time (hours)")
    ax2.set_ylabel("Modeled cost (model cost units)")
    ax2.set_title("Modeled Operational Cost")
    ax2.grid(True, alpha=0.3)
    mean_cost = np.mean(operational_costs)
    ax2.axhline(
        y=mean_cost,
        color="red",
        linestyle="--",
        label=f"Run mean: {mean_cost:.1f} model cost units",
    )
    ax2.legend()

    ax3 = axes[1, 0]
    total_containers = [kpi["total_containers"] for kpi in kpi_history]
    assigned_containers = [kpi["assigned_containers"] for kpi in kpi_history]
    ax3.fill_between(
        time_hours,
        0,
        total_containers,
        alpha=0.3,
        color="blue",
        label="Containers retained in simulation",
    )
    ax3.fill_between(
        time_hours,
        0,
        assigned_containers,
        alpha=0.5,
        color="green",
        label="Assigned containers retained",
    )
    ax3.set_xlabel("Simulated time (hours)")
    ax3.set_ylabel("Container records (count)")
    ax3.set_title("Container Records in the Simulation")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    metrics = [
        "Assignment\nefficiency",
        "Vehicle\nutilization",
        "On-time\nassignment",
    ]
    values = [
        average_kpis["transfer_efficiency"] * 100,
        average_kpis["vehicle_utilization"] * 100,
        average_kpis["on_time_assignment_rate"] * 100,
    ]
    bars = ax4.bar(
        metrics,
        values,
        color=["lightblue", "lightgreen", "lightcoral"],
        alpha=0.8,
    )
    ax4.set_ylabel("Mean rate (%)")
    ax4.set_title("Run-Level Mean Rates")
    ax4.set_ylim(0, 100)
    ax4.grid(True, axis="y", alpha=0.3)

    for bar, value in zip(bars, values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    plt.tight_layout()
    plt.show()
    return fig


fig = visualize_digital_twin_results(baseline_kpis, baseline_avg)

## Summary and Interpretation

### What the simulation demonstrates

The notebook connects several components in one simulated workflow:

- **Sensor generation:** Sensor objects produce records at their configured update intervals and can stop reporting after a simulated failure.
- **State updates:** Vehicle, terminal, and container states change as the simulation clock advances.
- **Demand and congestion analytics:** Recent simulated observations feed simple forecasting rules and bottleneck indicators.
- **Monitoring:** KPI values are calculated at each time step, and threshold-based alerts are recorded.
- **Visualization:** Plots show how the chosen KPIs evolve during the illustrative run.

### How to interpret the KPIs

Assignment efficiency is the share of container records currently retained in the simulation that have been assigned. Vehicle utilization compares total simulated load with total fleet capacity. Assignment waiting time is measured from a container's ready time to its assignment time, and the on-time assignment rate uses the model's 30-minute threshold. Modeled operational cost combines accumulated distance-weighted vehicle costs with a fixed labor term. These quantities depend on the definitions and assumptions in the code; in particular, they should not be confused with delivered-container performance or audited financial costs.

The plotted container counts describe records still present in the simulation. Assigned records are removed after the one-hour bookkeeping window, so these counts are not cumulative throughput. The forecasts and alerts are also illustrative outputs, not validated predictions or operational recommendations.

### Limitations and possible extensions

The simulation uses stochastic rules rather than calibrated physical or statistical models. It does not implement live sensor integration, a detailed transfer lifecycle, a validated demand model, or an optimization formulation. A practical study would need data definitions, validated assumptions, and an explicitly designed evaluation before interpreting the simulated metrics as evidence about a real terminal.

Natural extensions within this simulation framework include adding validated operational data, representing additional terminal processes, and defining a more detailed transfer lifecycle. These extensions would require new assumptions and validation; this notebook does not implement them.